# 04f — Siamese Euclidean V3

## Obiettivo

Addestrare e valutare la Euclidean Contrastive Loss utilizzando la stessa
architettura CNN V3 bias-free introdotta per il modello Cosine stabilizzato.

L'obiettivo è ottenere un confronto equo tra:

- Euclidean Contrastive Loss
- CosineEmbeddingLoss

mantenendo invariati encoder e pipeline di training.

## Configurazione fissata

- FCGR: k = 6
- Input: 1 × 64 × 64
- Dimensione embedding: 128
- Normalizzazione CNN: GroupNorm
- Bias nelle convoluzioni: False
- Normalizzazione embedding: L2
- Ottimizzatore: AdamW
- Learning rate: 5e-4
- AMP: attivo su CUDA
- Selezione del modello: validation ROC-AUC
- Euclidean margin iniziale: 1.25

Lo split del dataset utilizza `split_cluster` per evitare data leakage.

In [1]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================

from pathlib import Path

import copy
import gc
import random
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
)

from sklearn.metrics import (
    roc_auc_score,
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.12.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
# ============================================================
# CELL 3 — PATH E CONFIGURAZIONE
# ============================================================

import json


CURRENT_DIR = (
    Path.cwd()
    .resolve()
)


if CURRENT_DIR.name == "notebooks":

    PROJECT_ROOT = (
        CURRENT_DIR.parent
    )

else:

    PROJECT_ROOT = (
        CURRENT_DIR
    )


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


# Nuova cartella dedicata alla Euclidean V3
ARTIFACTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "siamese_euclidean_v3"
)

ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PAIR_CONFIG_PATH = (
    PROCESSED_DIR
    / "siamese_pair_config.json"
)


MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_primary_manifest.tsv"
)


VAL_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_val_pair_pool.tsv"
)


TEST_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_test_pair_pool.tsv"
)


with open(
    PAIR_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    PAIR_CONFIG = json.load(f)


DEFAULT_K = int(
    PAIR_CONFIG["k"]
)


ORIGINAL_BATCH_SIZE = int(
    PAIR_CONFIG["batch_size"]
)


# Manteniamo lo stesso batch usato
# nella Baseline C e nella Cosine V3.
BATCH_SIZE = 128


TRAIN_PAIRS_PER_EPOCH = int(
    PAIR_CONFIG[
        "train_pairs_per_epoch"
    ]
)


VAL_PAIRS = int(
    PAIR_CONFIG[
        "val_pairs"
    ]
)


TEST_PAIRS = int(
    PAIR_CONFIG[
        "test_pairs"
    ]
)


POSITIVE_PAIR_PROBABILITY = float(
    PAIR_CONFIG[
        "positive_probability"
    ]
)


RANDOM_STATE = int(
    PAIR_CONFIG[
        "random_state"
    ]
)


FCGR_MEMMAP_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{DEFAULT_K}.npy"
)


FCGR_INDEX_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{DEFAULT_K}_index.tsv"
)


print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "k:",
    DEFAULT_K
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Train pairs / epoch:",
    TRAIN_PAIRS_PER_EPOCH
)

print(
    "Validation pairs:",
    VAL_PAIRS
)

print(
    "Test pairs:",
    TEST_PAIRS
)

print(
    "Probabilità pair positive:",
    POSITIVE_PAIR_PROBABILITY
)

print(
    "Random state:",
    RANDOM_STATE
)

print(
    "Artifacts:",
    ARTIFACTS_DIR
)

Project root: D:\Daria\Desktop\eccdna_fcgr_siamese
k: 6
Batch size: 128
Train pairs / epoch: 50000
Validation pairs: 10000
Test pairs: 10000
Probabilità pair positive: 0.5
Random state: 42
Artifacts: D:\Daria\Desktop\eccdna_fcgr_siamese\artifacts\siamese_euclidean_v3


In [3]:
# ============================================================
# CELL 4 — RIPRODUCIBILITÀ E DEVICE
# ============================================================

def set_seed(
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


set_seed(
    RANDOM_STATE
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(
    "Device:",
    DEVICE
)


if DEVICE.type == "cuda":

    torch.backends.cudnn.benchmark = True

    torch.set_float32_matmul_precision(
        "high"
    )


print(
    "cuDNN benchmark:",
    torch.backends.cudnn.benchmark
)

Device: cuda
cuDNN benchmark: True


In [4]:
# ============================================================
# CELL 5 — CARICAMENTO METADATI E SPLIT
# ============================================================

metadata = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
    dtype={
        "id": str
    }
)


val_pair_pool = pd.read_csv(
    VAL_POOL_PATH,
    sep="\t",
    dtype={
        "id": str
    }
)


test_pair_pool = pd.read_csv(
    TEST_POOL_PATH,
    sep="\t",
    dtype={
        "id": str
    }
)


train_metadata = (
    metadata[
        metadata[
            "split_cluster"
        ] == "train"
    ]
    .copy()
)


# class_id deve essere intero
for dataframe in [
    train_metadata,
    val_pair_pool,
    test_pair_pool
]:

    dataframe[
        "class_id"
    ] = (
        dataframe[
            "class_id"
        ]
        .astype(int)
    )


print(
    "Train samples:",
    f"{len(train_metadata):,}"
)

print(
    "Validation pool:",
    f"{len(val_pair_pool):,}"
)

print(
    "Test pool:",
    f"{len(test_pair_pool):,}"
)

print(
    "Numero classi:",
    train_metadata[
        "class_id"
    ].nunique()
)


print()
print(
    "Distribuzione classi train:"
)

display(
    train_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
    .rename("n_samples")
    .to_frame()
)

Train samples: 126,265
Validation pool: 12,937
Test pool: 11,070
Numero classi: 18

Distribuzione classi train:


,n_samples
class_id,
0,10000
1,10000
2,10000
3,10000
4,10000
5,10000
6,10000
7,10000
8,10000


In [5]:
# ============================================================
# CELL 6 — CARICAMENTO FCGR MEMMAP
# ============================================================

fcgr_memmap = np.load(
    FCGR_MEMMAP_PATH,
    mmap_mode="r"
)


fcgr_index = pd.read_csv(
    FCGR_INDEX_PATH,
    sep="\t",
    dtype={
        "id": str
    }
)


id_to_fcgr_row = dict(
    zip(
        fcgr_index[
            "id"
        ],

        fcgr_index[
            "fcgr_row"
        ]
    )
)


print(
    "FCGR shape:",
    fcgr_memmap.shape
)

print(
    "FCGR dtype:",
    fcgr_memmap.dtype
)

print(
    "Sample indicizzati:",
    len(
        id_to_fcgr_row
    )
)


# ============================================================
# CONTROLLI
# ============================================================

assert (
    fcgr_memmap.dtype
    ==
    np.float32
)


assert (
    len(
        id_to_fcgr_row
    )
    ==
    fcgr_memmap.shape[0]
)


missing_train_ids = (
    set(
        train_metadata["id"]
    )
    -
    set(
        id_to_fcgr_row
    )
)


missing_val_ids = (
    set(
        val_pair_pool["id"]
    )
    -
    set(
        id_to_fcgr_row
    )
)


missing_test_ids = (
    set(
        test_pair_pool["id"]
    )
    -
    set(
        id_to_fcgr_row
    )
)


print()
print(
    "ID train senza FCGR:",
    len(missing_train_ids)
)

print(
    "ID validation senza FCGR:",
    len(missing_val_ids)
)

print(
    "ID test senza FCGR:",
    len(missing_test_ids)
)


assert len(missing_train_ids) == 0
assert len(missing_val_ids) == 0
assert len(missing_test_ids) == 0

FCGR shape: (150272, 64, 64)
FCGR dtype: float32
Sample indicizzati: 150272

ID train senza FCGR: 0
ID validation senza FCGR: 0
ID test senza FCGR: 0


In [19]:
# ============================================================
# CELL 7 — DATASET SIAMESE OTTIMIZZATO
# ============================================================

class SiamesePairDataset(
    Dataset
):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row,
        pairs_per_epoch,
        positive_probability=0.5,
        seed=42,
        deterministic=False
    ):

        self.fcgr_memmap = (
            fcgr_memmap
        )

        self.pairs_per_epoch = int(
            pairs_per_epoch
        )

        self.positive_probability = float(
            positive_probability
        )

        self.seed = int(
            seed
        )

        self.deterministic = bool(
            deterministic
        )


        # ====================================================
        # CONVERSIONE ID -> FCGR ROW
        # FATTA UNA SOLA VOLTA
        # ====================================================

        metadata_local = (
            metadata[
                [
                    "id",
                    "class_id"
                ]
            ]
            .copy()
            .reset_index(drop=True)
        )


        metadata_local[
            "fcgr_row"
        ] = (

            metadata_local[
                "id"
            ]
            .map(
                id_to_row
            )
            .astype(
                np.int64
            )
        )


        # ====================================================
        # CLASS -> ARRAY DI FCGR ROW
        # ====================================================

        self.class_to_rows = {

            int(class_id):

            group[
                "fcgr_row"
            ]
            .to_numpy(
                dtype=np.int64
            )

            for class_id, group
            in metadata_local.groupby(
                "class_id"
            )
        }


        self.classes = np.array(

            sorted(
                self.class_to_rows.keys()
            ),

            dtype=np.int64
        )


        for class_id in self.classes:

            if (
                len(
                    self.class_to_rows[
                        int(class_id)
                    ]
                )
                < 2
            ):

                raise ValueError(
                    f"Classe {class_id} "
                    "con meno di 2 sample."
                )


        # ====================================================
        # RNG TRAIN
        # ====================================================

        self.rng = (
            np.random.default_rng(
                self.seed
            )
        )


        # ====================================================
        # VALIDATION / TEST:
        # PREGENERAZIONE DELLE COPPIE
        #
        # Così non rigeneriamo le pair ad ogni epoch.
        # ====================================================

        self.fixed_pairs = None


        if self.deterministic:

            rng = (
                np.random.default_rng(
                    self.seed
                )
            )


            row1_array = np.empty(
                self.pairs_per_epoch,
                dtype=np.int64
            )

            row2_array = np.empty(
                self.pairs_per_epoch,
                dtype=np.int64
            )

            target_array = np.empty(
                self.pairs_per_epoch,
                dtype=np.float32
            )


            for i in range(
                self.pairs_per_epoch
            ):

                (
                    row1,
                    row2,
                    target
                ) = self._sample_pair(
                    rng
                )


                row1_array[i] = row1
                row2_array[i] = row2
                target_array[i] = target


            self.fixed_pairs = (
                row1_array,
                row2_array,
                target_array
            )


    def __len__(
        self
    ):

        return (
            self.pairs_per_epoch
        )


    def _sample_pair(
        self,
        rng
    ):

        # ====================================================
        # PRIMA CLASSE
        # ====================================================

        class1_idx = int(
            rng.integers(
                0,
                len(self.classes)
            )
        )

        class1 = int(
            self.classes[
                class1_idx
            ]
        )


        rows1 = (
            self.class_to_rows[
                class1
            ]
        )


        index1 = int(
            rng.integers(
                0,
                len(rows1)
            )
        )


        row1 = int(
            rows1[
                index1
            ]
        )


        # ====================================================
        # POSITIVE PAIR
        # ====================================================

        if (
            rng.random()
            <
            self.positive_probability
        ):

            # Scegliamo direttamente un altro indice
            # senza while loop.

            index2 = int(
                rng.integers(
                    0,
                    len(rows1) - 1
                )
            )


            if (
                index2
                >=
                index1
            ):

                index2 += 1


            row2 = int(
                rows1[
                    index2
                ]
            )


            target = (
                1.0
            )


        # ====================================================
        # NEGATIVE PAIR
        # ====================================================

        else:

            # Scegliamo una classe diversa senza
            # costruire ogni volta possible_classes.

            class2_idx = int(
                rng.integers(
                    0,
                    len(self.classes) - 1
                )
            )


            if (
                class2_idx
                >=
                class1_idx
            ):

                class2_idx += 1


            class2 = int(
                self.classes[
                    class2_idx
                ]
            )


            rows2 = (
                self.class_to_rows[
                    class2
                ]
            )


            row2 = int(

                rows2[

                    rng.integers(
                        0,
                        len(rows2)
                    )

                ]

            )


            target = (
                0.0
            )


        return (
            row1,
            row2,
            target
        )


    def _load_fcgr_row(
        self,
        row
    ):

        fcgr = np.array(

            self.fcgr_memmap[
                row
            ],

            dtype=np.float32,

            copy=True
        )


        return (
            torch.from_numpy(
                fcgr
            )
            .unsqueeze(0)
        )


    def __getitem__(
        self,
        index
    ):

        # ====================================================
        # FIXED VALIDATION / TEST
        # ====================================================

        if self.deterministic:

            (
                rows1,
                rows2,
                targets
            ) = (
                self.fixed_pairs
            )


            row1 = int(
                rows1[index]
            )

            row2 = int(
                rows2[index]
            )

            target = float(
                targets[index]
            )


        # ====================================================
        # DYNAMIC TRAIN
        # ====================================================

        else:

            (
                row1,
                row2,
                target
            ) = self._sample_pair(
                self.rng
            )


        x1 = (
            self._load_fcgr_row(
                row1
            )
        )


        x2 = (
            self._load_fcgr_row(
                row2
            )
        )


        return {

            "x1":
                x1,

            "x2":
                x2,

            "target":
                torch.tensor(
                    target,
                    dtype=torch.float32
                )
        }

In [20]:
# ============================================================
# CELL 8 — DATASET E DATALOADER
# ============================================================

train_dataset = SiamesePairDataset(

    metadata=
        train_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row,

    pairs_per_epoch=
        TRAIN_PAIRS_PER_EPOCH,

    positive_probability=
        POSITIVE_PAIR_PROBABILITY,

    seed=
        RANDOM_STATE,

    deterministic=False
)


val_dataset = SiamesePairDataset(

    metadata=
        val_pair_pool,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row,

    pairs_per_epoch=
        VAL_PAIRS,

    positive_probability=
        POSITIVE_PAIR_PROBABILITY,

    seed=
        RANDOM_STATE + 10_000,

    deterministic=True
)


test_dataset = SiamesePairDataset(

    metadata=
        test_pair_pool,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row,

    pairs_per_epoch=
        TEST_PAIRS,

    positive_probability=
        POSITIVE_PAIR_PROBABILITY,

    seed=
        RANDOM_STATE + 20_000,

    deterministic=True
)


NUM_WORKERS = 0


train_loader = DataLoader(

    train_dataset,

    batch_size=
        BATCH_SIZE,

    shuffle=False,

    num_workers=
        NUM_WORKERS,

    pin_memory=
        torch.cuda.is_available()
)


val_loader = DataLoader(

    val_dataset,

    batch_size=
        BATCH_SIZE,

    shuffle=False,

    num_workers=
        NUM_WORKERS,

    pin_memory=
        torch.cuda.is_available()
)


test_loader = DataLoader(

    test_dataset,

    batch_size=
        BATCH_SIZE,

    shuffle=False,

    num_workers=
        NUM_WORKERS,

    pin_memory=
        torch.cuda.is_available()
)


print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "Test batches:",
    len(test_loader)
)

Train batches: 391
Validation batches: 79
Test batches: 79


In [21]:
# ============================================================
# CELL 9 — CONTROLLO DI UN BATCH
# ============================================================

batch = next(
    iter(
        train_loader
    )
)


print(
    "x1:",
    batch["x1"].shape,
    batch["x1"].dtype
)

print(
    "x2:",
    batch["x2"].shape,
    batch["x2"].dtype
)

print(
    "target:",
    batch["target"].shape,
    batch["target"].dtype
)


print()

print(
    "Positive:",
    (
        batch[
            "target"
        ] == 1
    ).sum().item()
)

print(
    "Negative:",
    (
        batch[
            "target"
        ] == 0
    ).sum().item()
)


assert (
    batch["x1"].shape[1:]
    ==
    (
        1,
        64,
        64
    )
)

assert (
    batch["x1"].dtype
    ==
    torch.float32
)

assert (
    set(
        batch[
            "target"
        ].unique().tolist()
    )
    <=
    {
        0.0,
        1.0
    }
)

x1: torch.Size([128, 1, 64, 64]) torch.float32
x2: torch.Size([128, 1, 64, 64]) torch.float32
target: torch.Size([128]) torch.float32

Positive: 68
Negative: 60


## CNN Encoder V3 bias-free

Per ottenere un confronto corretto con il modello Cosine stabilizzato viene
utilizzata la stessa architettura CNN V3.

Durante gli esperimenti con `CosineEmbeddingLoss` è stato osservato che le FCGR
normalizzate sono molto sparse e hanno valori medi molto piccoli.

I bias iniziali delle convoluzioni risultavano molto più grandi del segnale
prodotto dalle FCGR e introducevano quindi una forte componente comune tra i
sample.

Questo portava gli embedding iniziali a essere quasi paralleli.

La versione V3 utilizza quindi:

`Conv2d(..., bias=False)`

in tutti i layer convoluzionali.

Tutte le altre caratteristiche principali dell'encoder rimangono invariate:

- GroupNorm;
- ReLU;
- MaxPooling;
- Adaptive Average Pooling;
- embedding a 128 dimensioni;
- normalizzazione L2 finale.

In [9]:
# ============================================================
# CELL 11 — CNN ENCODER V3 BIAS-FREE
# ============================================================

EMBEDDING_DIM = 128


class FCGRCNNEncoderV3(
    nn.Module
):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()


        self.features = nn.Sequential(

            # =================================================
            # 64 x 64
            # =================================================

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                num_groups=8,
                num_channels=32
            ),

            nn.ReLU(
                inplace=True
            ),


            nn.Conv2d(
                32,
                32,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                kernel_size=2
            ),


            # =================================================
            # 32 x 32
            # =================================================

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                num_groups=8,
                num_channels=64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                kernel_size=2
            ),


            # =================================================
            # 16 x 16
            # =================================================

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                num_groups=8,
                num_channels=128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                kernel_size=2
            ),


            # =================================================
            # 8 x 8
            # =================================================

            nn.Conv2d(
                128,
                128,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                num_groups=8,
                num_channels=128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.AdaptiveAvgPool2d(
                (
                    4,
                    4
                )
            )
        )


        self.embedding_head = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Linear(
                256,
                embedding_dim
            )
        )


    def forward(
        self,
        x
    ):

        x = (
            self.features(
                x
            )
        )

        z = (
            self.embedding_head(
                x
            )
        )

        # Embedding L2-normalizzato.
        z = F.normalize(
            z,
            p=2,
            dim=1,
            eps=1e-8
        )

        return z

In [10]:
# ============================================================
# CELL 12 — SIAMESE NETWORK V3 + SANITY CHECK
# ============================================================

class SiameseNetworkEuclideanV3(
    nn.Module
):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()

        self.encoder = (
            FCGRCNNEncoderV3(
                embedding_dim=
                    embedding_dim
            )
        )


    def forward(
        self,
        x1,
        x2
    ):

        batch_size = (
            x1.shape[0]
        )

        # Un unico forward dell'encoder
        # per entrambe le metà della coppia.
        x = torch.cat(
            [
                x1,
                x2
            ],
            dim=0
        )

        z = (
            self.encoder(
                x
            )
        )

        z1 = (
            z[
                :batch_size
            ]
        )

        z2 = (
            z[
                batch_size:
            ]
        )

        return (
            z1,
            z2
        )


# ============================================================
# CREAZIONE TEMPORANEA DEL MODELLO
# ============================================================

set_seed(
    RANDOM_STATE
)


model_check = (
    SiameseNetworkEuclideanV3(
        embedding_dim=
            EMBEDDING_DIM
    )
    .to(
        DEVICE
    )
)


# ============================================================
# VERIFICA CHE TUTTE LE CONV SIANO BIAS-FREE
# ============================================================

conv_layers = [
    module

    for module
    in model_check.modules()

    if isinstance(
        module,
        nn.Conv2d
    )
]


print(
    "Numero Conv2d:",
    len(conv_layers)
)


for i, conv in enumerate(
    conv_layers,
    start=1
):

    print(
        f"Conv {i}:",
        "bias =",
        conv.bias
    )

    assert (
        conv.bias
        is None
    )


# ============================================================
# FORWARD TEST
# ============================================================

x1_check = (
    batch[
        "x1"
    ][:8]
    .to(
        DEVICE
    )
)


x2_check = (
    batch[
        "x2"
    ][:8]
    .to(
        DEVICE
    )
)


model_check.eval()


with torch.no_grad():

    (
        z1_check,
        z2_check
    ) = model_check(
        x1_check,
        x2_check
    )


print()

print(
    "z1 shape:",
    z1_check.shape
)

print(
    "z2 shape:",
    z2_check.shape
)


print(
    "Norma media z1:",
    z1_check.norm(
        p=2,
        dim=1
    ).mean().item()
)

print(
    "Norma media z2:",
    z2_check.norm(
        p=2,
        dim=1
    ).mean().item()
)


assert (
    z1_check.shape
    ==
    (
        8,
        EMBEDDING_DIM
    )
)

assert torch.allclose(

    z1_check.norm(
        p=2,
        dim=1
    ),

    torch.ones(
        8,
        device=DEVICE
    ),

    atol=1e-4
)


del model_check
del x1_check
del x2_check
del z1_check
del z2_check

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

Numero Conv2d: 5
Conv 1: bias = None
Conv 2: bias = None
Conv 3: bias = None
Conv 4: bias = None
Conv 5: bias = None

z1 shape: torch.Size([8, 128])
z2 shape: torch.Size([8, 128])
Norma media z1: 1.0
Norma media z2: 1.0


In [11]:
# ============================================================
# CELL 13 — EUCLIDEAN CONTRASTIVE LOSS
# ============================================================

EUCLIDEAN_MARGIN = 1.25


class EuclideanContrastiveLoss(nn.Module):

    def __init__(
        self,
        margin=1.25
    ):

        super().__init__()

        self.margin = float(
            margin
        )


    def forward(
        self,
        z1,
        z2,
        target
    ):

        # Distanza euclidea tra embedding L2-normalizzati
        distances = F.pairwise_distance(
            z1,
            z2,
            p=2,
            eps=1e-8
        )


        # Coppie positive:
        # minimizziamo la distanza
        positive_loss = (
            target
            *
            distances.pow(2)
        )


        # Coppie negative:
        # penalizziamo solo se distanza < margin
        negative_loss = (
            (1.0 - target)
            *
            F.relu(
                self.margin
                -
                distances
            ).pow(2)
        )


        loss = (
            positive_loss
            +
            negative_loss
        ).mean()


        return (
            loss,
            distances
        )


criterion_check = (
    EuclideanContrastiveLoss(
        margin=
            EUCLIDEAN_MARGIN
    )
)


print(
    "Euclidean margin:",
    criterion_check.margin
)

Euclidean margin: 1.25


In [12]:
# ============================================================
# CELL 14 — METRICHE EUCLIDEAN
# ============================================================

def compute_euclidean_metrics(
    targets,
    distances
):

    targets = np.asarray(
        targets,
        dtype=np.int64
    )

    distances = np.asarray(
        distances,
        dtype=np.float64
    )


    positive_distances = (
        distances[
            targets == 1
        ]
    )


    negative_distances = (
        distances[
            targets == 0
        ]
    )


    d_pos = float(
        positive_distances.mean()
    )


    d_neg = float(
        negative_distances.mean()
    )


    gap = float(
        d_neg
        -
        d_pos
    )


    pooled_std = np.sqrt(
        0.5
        *
        (
            positive_distances.var()
            +
            negative_distances.var()
        )
        +
        1e-12
    )


    d_prime = float(
        gap
        /
        pooled_std
    )


    # Più piccola è la distanza,
    # maggiore deve essere lo score di similarità.
    similarity_scores = (
        -distances
    )


    roc_auc = float(
        roc_auc_score(
            targets,
            similarity_scores
        )
    )


    return {

        "roc_auc":
            roc_auc,

        "d_pos":
            d_pos,

        "d_neg":
            d_neg,

        "gap":
            gap,

        "d_prime":
            d_prime
    }

In [13]:
# ============================================================
# CELL 15 — CONFIGURAZIONE TRAINING
# ============================================================

LEARNING_RATE = 5e-4

WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 50

EARLY_STOPPING_PATIENCE = 10

MIN_DELTA = 1e-4


AMP_ENABLED = (
    DEVICE.type == "cuda"
)


def build_euclidean_v3_components(
    margin=1.25
):

    set_seed(
        RANDOM_STATE
    )


    model = (
        SiameseNetworkEuclideanV3(
            embedding_dim=
                EMBEDDING_DIM
        )
        .to(
            DEVICE
        )
    )


    # --------------------------------------------------------
    # AdamW fused quando disponibile
    # --------------------------------------------------------

    fused_adamw = False


    if DEVICE.type == "cuda":

        try:

            optimizer = torch.optim.AdamW(

                model.parameters(),

                lr=
                    LEARNING_RATE,

                weight_decay=
                    WEIGHT_DECAY,

                fused=True
            )

            fused_adamw = True


        except (
            TypeError,
            RuntimeError
        ):

            optimizer = torch.optim.AdamW(

                model.parameters(),

                lr=
                    LEARNING_RATE,

                weight_decay=
                    WEIGHT_DECAY
            )


    else:

        optimizer = torch.optim.AdamW(

            model.parameters(),

            lr=
                LEARNING_RATE,

            weight_decay=
                WEIGHT_DECAY
        )


    scaler = torch.cuda.amp.GradScaler(
        enabled=
            AMP_ENABLED
    )


    criterion = (
        EuclideanContrastiveLoss(
            margin=
                margin
        )
    )


    return (
        model,
        optimizer,
        scaler,
        criterion,
        fused_adamw
    )


print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "AMP:",
    AMP_ENABLED
)

print(
    "Max epochs:",
    MAX_EPOCHS
)

print(
    "Early stopping:",
    EARLY_STOPPING_PATIENCE
)

Learning rate: 0.0005
Weight decay: 0.0001
AMP: True
Max epochs: 50
Early stopping: 10


In [14]:
# ============================================================
# CELL 16 — RUN EPOCH
# ============================================================

def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
    scaler=None,
    collect_outputs=False
):

    is_training = (
        optimizer
        is not None
    )


    if is_training:

        model.train()

    else:

        model.eval()


    total_loss = 0.0
    total_pairs = 0


    all_targets = []
    all_distances = []


    start_time = time.perf_counter()


    for batch in loader:

        x1 = (
            batch["x1"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        x2 = (
            batch["x2"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        targets = (
            batch["target"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        batch_size = (
            targets.shape[0]
        )


        # ====================================================
        # TRAIN
        # ====================================================

        if is_training:

            optimizer.zero_grad(
                set_to_none=True
            )


            with torch.autocast(

                device_type=
                    DEVICE.type,

                dtype=
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16,

                enabled=
                    AMP_ENABLED

            ):

                (
                    z1,
                    z2
                ) = model(
                    x1,
                    x2
                )


                (
                    loss,
                    distances
                ) = criterion(
                    z1,
                    z2,
                    targets
                )


            if (
                scaler
                is not None
                and
                AMP_ENABLED
            ):

                scaler.scale(
                    loss
                ).backward()


                scaler.step(
                    optimizer
                )


                scaler.update()


            else:

                loss.backward()

                optimizer.step()


        # ====================================================
        # VALIDATION / TEST
        # ====================================================

        else:

            with torch.no_grad():

                with torch.autocast(

                    device_type=
                        DEVICE.type,

                    dtype=
                        torch.float16
                        if DEVICE.type == "cuda"
                        else torch.bfloat16,

                    enabled=
                        AMP_ENABLED

                ):

                    (
                        z1,
                        z2
                    ) = model(
                        x1,
                        x2
                    )


                    (
                        loss,
                        distances
                    ) = criterion(
                        z1,
                        z2,
                        targets
                    )


        # ====================================================
        # ACCUMULO
        # ====================================================

        total_loss += (
            loss.detach().item()
            *
            batch_size
        )


        total_pairs += (
            batch_size
        )


        all_targets.append(
            targets
            .detach()
            .cpu()
            .numpy()
        )


        all_distances.append(
            distances
            .detach()
            .float()
            .cpu()
            .numpy()
        )


    elapsed = (
        time.perf_counter()
        -
        start_time
    )


    all_targets = np.concatenate(
        all_targets
    )


    all_distances = np.concatenate(
        all_distances
    )


    metric_values = (
        compute_euclidean_metrics(
            all_targets,
            all_distances
        )
    )


    metrics = {

        "loss":
            total_loss
            /
            total_pairs,

        "roc_auc":
            metric_values[
                "roc_auc"
            ],

        "d_pos":
            metric_values[
                "d_pos"
            ],

        "d_neg":
            metric_values[
                "d_neg"
            ],

        "gap":
            metric_values[
                "gap"
            ],

        "d_prime":
            metric_values[
                "d_prime"
            ],

        "seconds":
            elapsed,

        "pairs_per_second":
            total_pairs
            /
            max(
                elapsed,
                1e-12
            )
    }


    if collect_outputs:

        return (
            metrics,
            all_targets,
            all_distances
        )


    return metrics

In [15]:
# ============================================================
# CELL 17 — SMOKE TEST EUCLIDEAN V3
# ============================================================

SMOKE_EPOCHS = 5


(
    smoke_model,
    smoke_optimizer,
    smoke_scaler,
    smoke_criterion,
    smoke_fused_adamw
) = build_euclidean_v3_components(
    margin=
        EUCLIDEAN_MARGIN
)


print("=" * 70)
print("EUCLIDEAN V3 — SMOKE TEST")
print("=" * 70)

print(
    "Margin:",
    EUCLIDEAN_MARGIN
)

print(
    "Fused AdamW:",
    smoke_fused_adamw
)

print(
    "AMP:",
    AMP_ENABLED
)

print()


for epoch in range(
    1,
    SMOKE_EPOCHS + 1
):

    train_metrics = run_epoch(

        model=
            smoke_model,

        loader=
            train_loader,

        criterion=
            smoke_criterion,

        optimizer=
            smoke_optimizer,

        scaler=
            smoke_scaler,

        collect_outputs=False
    )


    val_metrics = run_epoch(

        model=
            smoke_model,

        loader=
            val_loader,

        criterion=
            smoke_criterion,

        optimizer=None,

        scaler=None,

        collect_outputs=False
    )


    print(

        f"Epoch {epoch:02d}/{SMOKE_EPOCHS}"

        f" | train loss "
        f"{train_metrics['loss']:.5f}"

        f" | val loss "
        f"{val_metrics['loss']:.5f}"

        f" | val AUC "
        f"{val_metrics['roc_auc']:.5f}"

        f" | d+ "
        f"{val_metrics['d_pos']:.4f}"

        f" | d- "
        f"{val_metrics['d_neg']:.4f}"

        f" | gap "
        f"{val_metrics['gap']:.4f}"

        f" | d' "
        f"{val_metrics['d_prime']:.3f}"

        f" | "
        f"{train_metrics['seconds'] + val_metrics['seconds']:.1f}s"
    )

C:\Users\simon\AppData\Local\Temp\ipykernel_5232\659637786.py:99: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(


EUCLIDEAN V3 — SMOKE TEST
Margin: 1.25
Fused AdamW: True
AMP: True

Epoch 01/5 | train loss 0.38603 | val loss 0.37219 | val AUC 0.63301 | d+ 0.5635 | d- 0.6554 | gap 0.0919 | d' 0.481 | 126.2s
Epoch 02/5 | train loss 0.37242 | val loss 0.36740 | val AUC 0.64894 | d+ 0.5268 | d- 0.6259 | gap 0.0991 | d' 0.540 | 115.0s
Epoch 03/5 | train loss 0.36748 | val loss 0.36213 | val AUC 0.65722 | d+ 0.5507 | d- 0.6682 | gap 0.1175 | d' 0.579 | 114.5s
Epoch 04/5 | train loss 0.36511 | val loss 0.36080 | val AUC 0.66055 | d+ 0.5878 | d- 0.7051 | gap 0.1173 | d' 0.589 | 114.5s
Epoch 05/5 | train loss 0.36506 | val loss 0.35831 | val AUC 0.66535 | d+ 0.5643 | d- 0.6744 | gap 0.1101 | d' 0.602 | 113.8s


In [22]:
# ============================================================
# CELL 18 — BENCHMARK DATALOADER
# ============================================================

N_BENCH_BATCHES = 30

start = time.perf_counter()

n_pairs = 0

for i, batch_bench in enumerate(train_loader):

    n_pairs += batch_bench["target"].shape[0]

    if i + 1 >= N_BENCH_BATCHES:
        break

elapsed = time.perf_counter() - start


print("=" * 70)
print("DATALOADER BENCHMARK")
print("=" * 70)

print(
    "Batch:",
    N_BENCH_BATCHES
)

print(
    "Pairs:",
    n_pairs
)

print(
    "Tempo:",
    f"{elapsed:.2f} s"
)

print(
    "Pairs/s:",
    f"{n_pairs / elapsed:.1f}"
)

DATALOADER BENCHMARK
Batch: 30
Pairs: 3840
Tempo: 0.20 s
Pairs/s: 19554.4


In [17]:
# ============================================================
# CELL 19 — BENCHMARK PURO GPU
# ============================================================

(
    bench_model,
    bench_optimizer,
    bench_scaler,
    bench_criterion,
    _
) = build_euclidean_v3_components(
    margin=EUCLIDEAN_MARGIN
)


batch_gpu = next(
    iter(train_loader)
)


x1_gpu = (
    batch_gpu["x1"]
    .to(
        DEVICE,
        non_blocking=True
    )
)


x2_gpu = (
    batch_gpu["x2"]
    .to(
        DEVICE,
        non_blocking=True
    )
)


target_gpu = (
    batch_gpu["target"]
    .to(
        DEVICE,
        non_blocking=True
    )
)


# ============================================================
# WARM-UP
# ============================================================

bench_model.train()

for _ in range(5):

    bench_optimizer.zero_grad(
        set_to_none=True
    )

    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=AMP_ENABLED
    ):

        z1, z2 = bench_model(
            x1_gpu,
            x2_gpu
        )

        loss, _ = bench_criterion(
            z1,
            z2,
            target_gpu
        )

    bench_scaler.scale(
        loss
    ).backward()

    bench_scaler.step(
        bench_optimizer
    )

    bench_scaler.update()


torch.cuda.synchronize()


# ============================================================
# BENCHMARK
# ============================================================

N_GPU_STEPS = 30

start = time.perf_counter()


for _ in range(N_GPU_STEPS):

    bench_optimizer.zero_grad(
        set_to_none=True
    )

    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=AMP_ENABLED
    ):

        z1, z2 = bench_model(
            x1_gpu,
            x2_gpu
        )

        loss, _ = bench_criterion(
            z1,
            z2,
            target_gpu
        )

    bench_scaler.scale(
        loss
    ).backward()

    bench_scaler.step(
        bench_optimizer
    )

    bench_scaler.update()


torch.cuda.synchronize()


elapsed = (
    time.perf_counter()
    -
    start
)


seconds_per_batch = (
    elapsed
    /
    N_GPU_STEPS
)


print("=" * 70)
print("GPU BENCHMARK")
print("=" * 70)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Steps:",
    N_GPU_STEPS
)

print(
    "Tempo totale:",
    f"{elapsed:.2f} s"
)

print(
    "Secondi/batch:",
    f"{seconds_per_batch:.4f}"
)

print(
    "Pairs/s GPU:",
    f"{BATCH_SIZE / seconds_per_batch:.1f}"
)


del bench_model
del bench_optimizer
del bench_scaler
del bench_criterion

del batch_gpu
del x1_gpu
del x2_gpu
del target_gpu

gc.collect()

torch.cuda.empty_cache()

C:\Users\simon\AppData\Local\Temp\ipykernel_5232\659637786.py:99: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(


GPU BENCHMARK
Batch size: 128
Steps: 30
Tempo totale: 1.20 s
Secondi/batch: 0.0401
Pairs/s GPU: 3195.3


In [18]:
# ============================================================
# CELL 20 — GPU STATUS
# ============================================================

import subprocess

result = subprocess.run(
    [
        "nvidia-smi"
    ],
    capture_output=True,
    text=True
)

print(
    result.stdout
)

Wed Sep  2 09:00:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 592.27                 Driver Version: 592.27         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   49C    P8             11W /   85W |    3208MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [23]:
# ============================================================
# CELL 21 — FRESH DATALOADERS FOR FULL TRAINING
# ============================================================

set_seed(
    RANDOM_STATE
)


train_dataset = SiamesePairDataset(

    metadata=train_metadata,

    fcgr_memmap=fcgr_memmap,

    id_to_row=id_to_fcgr_row,

    pairs_per_epoch=TRAIN_PAIRS_PER_EPOCH,

    positive_probability=POSITIVE_PAIR_PROBABILITY,

    seed=RANDOM_STATE,

    deterministic=False
)


val_dataset = SiamesePairDataset(

    metadata=val_pair_pool,

    fcgr_memmap=fcgr_memmap,

    id_to_row=id_to_fcgr_row,

    pairs_per_epoch=VAL_PAIRS,

    positive_probability=POSITIVE_PAIR_PROBABILITY,

    seed=RANDOM_STATE + 10_000,

    deterministic=True
)


test_dataset = SiamesePairDataset(

    metadata=test_pair_pool,

    fcgr_memmap=fcgr_memmap,

    id_to_row=id_to_fcgr_row,

    pairs_per_epoch=TEST_PAIRS,

    positive_probability=POSITIVE_PAIR_PROBABILITY,

    seed=RANDOM_STATE + 20_000,

    deterministic=True
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


print("Fresh loaders creati.")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Fresh loaders creati.
Train batches: 391
Val batches: 79
Test batches: 79


In [24]:
# ============================================================
# CELL 22 — FULL TRAINING EUCLIDEAN V3
# ============================================================

BEST_CHECKPOINT_PATH = (
    ARTIFACTS_DIR
    / "euclidean_v3_margin_1p25_best.pt"
)


HISTORY_PATH = (
    ARTIFACTS_DIR
    / "euclidean_v3_margin_1p25_history.tsv"
)


(
    euclidean_v3_model,
    euclidean_v3_optimizer,
    euclidean_v3_scaler,
    euclidean_v3_criterion,
    euclidean_v3_fused
) = build_euclidean_v3_components(
    margin=EUCLIDEAN_MARGIN
)


print("=" * 72)
print("EUCLIDEAN V3 — FULL TRAINING")
print("=" * 72)

print(
    "Margin:",
    EUCLIDEAN_MARGIN
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Fused AdamW:",
    euclidean_v3_fused
)

print(
    "AMP:",
    AMP_ENABLED
)

print(
    "Max epochs:",
    MAX_EPOCHS
)

print(
    "Early stopping patience:",
    EARLY_STOPPING_PATIENCE
)

print()


history_rows = []

best_val_auc = -np.inf

best_epoch = 0

epochs_without_improvement = 0


for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    # ========================================================
    # TRAIN
    # ========================================================

    train_metrics = run_epoch(

        model=euclidean_v3_model,

        loader=train_loader,

        criterion=euclidean_v3_criterion,

        optimizer=euclidean_v3_optimizer,

        scaler=euclidean_v3_scaler,

        collect_outputs=False
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    val_metrics = run_epoch(

        model=euclidean_v3_model,

        loader=val_loader,

        criterion=euclidean_v3_criterion,

        optimizer=None,

        scaler=None,

        collect_outputs=False
    )


    val_auc = float(
        val_metrics[
            "roc_auc"
        ]
    )


    # ========================================================
    # CHECKPOINT
    # ========================================================

    improved = (
        val_auc
        >
        best_val_auc
        +
        MIN_DELTA
    )


    if improved:

        best_val_auc = (
            val_auc
        )

        best_epoch = (
            epoch
        )

        epochs_without_improvement = 0


        torch.save(
            {
                "model_state_dict":
                    euclidean_v3_model.state_dict(),

                "margin":
                    float(
                        EUCLIDEAN_MARGIN
                    ),

                "embedding_dim":
                    int(
                        EMBEDDING_DIM
                    ),

                "best_epoch":
                    int(
                        best_epoch
                    ),

                "best_val_auc":
                    float(
                        best_val_auc
                    ),

                "architecture":
                    "SiameseNetworkEuclideanV3",

                "conv_bias":
                    False,

                "learning_rate":
                    float(
                        LEARNING_RATE
                    ),

                "weight_decay":
                    float(
                        WEIGHT_DECAY
                    ),

                "random_state":
                    int(
                        RANDOM_STATE
                    )
            },

            BEST_CHECKPOINT_PATH
        )


    else:

        epochs_without_improvement += 1


    # ========================================================
    # HISTORY
    # ========================================================

    history_rows.append(
        {
            "epoch":
                epoch,

            "train_loss":
                train_metrics[
                    "loss"
                ],

            "train_auc":
                train_metrics[
                    "roc_auc"
                ],

            "val_loss":
                val_metrics[
                    "loss"
                ],

            "val_auc":
                val_auc,

            "d_pos":
                val_metrics[
                    "d_pos"
                ],

            "d_neg":
                val_metrics[
                    "d_neg"
                ],

            "gap":
                val_metrics[
                    "gap"
                ],

            "d_prime":
                val_metrics[
                    "d_prime"
                ],

            "train_seconds":
                train_metrics[
                    "seconds"
                ],

            "val_seconds":
                val_metrics[
                    "seconds"
                ]
        }
    )


    marker = (
        " *BEST*"
        if improved
        else ""
    )


    total_seconds = (
        train_metrics[
            "seconds"
        ]
        +
        val_metrics[
            "seconds"
        ]
    )


    print(

        f"Epoch {epoch:02d}/{MAX_EPOCHS}"

        f" | train loss "
        f"{train_metrics['loss']:.5f}"

        f" | val loss "
        f"{val_metrics['loss']:.5f}"

        f" | val AUC "
        f"{val_auc:.5f}"

        f" | d+ "
        f"{val_metrics['d_pos']:.4f}"

        f" | d- "
        f"{val_metrics['d_neg']:.4f}"

        f" | gap "
        f"{val_metrics['gap']:.4f}"

        f" | d' "
        f"{val_metrics['d_prime']:.3f}"

        f" | "
        f"{total_seconds:.1f}s"

        f"{marker}"
    )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if (
        epochs_without_improvement
        >=
        EARLY_STOPPING_PATIENCE
    ):

        print()
        print(
            f"Early stopping at epoch {epoch}"
        )

        break


# ============================================================
# SAVE HISTORY
# ============================================================

euclidean_v3_history = (
    pd.DataFrame(
        history_rows
    )
)


euclidean_v3_history.to_csv(
    HISTORY_PATH,
    sep="\t",
    index=False
)


# ============================================================
# RESTORE BEST CHECKPOINT
# ============================================================

best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE
)


euclidean_v3_model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)


euclidean_v3_model.eval()


print()
print("=" * 72)
print("TRAINING COMPLETE")
print("=" * 72)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation ROC-AUC:",
    f"{best_val_auc:.6f}"
)

print(
    "Checkpoint:",
    BEST_CHECKPOINT_PATH
)

print(
    "History:",
    HISTORY_PATH
)

C:\Users\simon\AppData\Local\Temp\ipykernel_5232\659637786.py:99: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(


EUCLIDEAN V3 — FULL TRAINING
Margin: 1.25
Learning rate: 0.0005
Fused AdamW: True
AMP: True
Max epochs: 50
Early stopping patience: 10

Epoch 01/50 | train loss 0.38611 | val loss 0.37295 | val AUC 0.63405 | d+ 0.5591 | d- 0.6535 | gap 0.0944 | d' 0.481 | 21.9s *BEST*
Epoch 02/50 | train loss 0.37212 | val loss 0.37304 | val AUC 0.64740 | d+ 0.4935 | d- 0.5936 | gap 0.1001 | d' 0.538 | 20.9s *BEST*
Epoch 03/50 | train loss 0.36855 | val loss 0.36362 | val AUC 0.65498 | d+ 0.5653 | d- 0.6835 | gap 0.1182 | d' 0.568 | 20.7s *BEST*
Epoch 04/50 | train loss 0.36456 | val loss 0.35972 | val AUC 0.66252 | d+ 0.5711 | d- 0.6858 | gap 0.1147 | d' 0.592 | 102.0s *BEST*
Epoch 05/50 | train loss 0.36443 | val loss 0.36017 | val AUC 0.66078 | d+ 0.5681 | d- 0.6743 | gap 0.1062 | d' 0.584 | 20.9s
Epoch 06/50 | train loss 0.36499 | val loss 0.36293 | val AUC 0.65562 | d+ 0.5668 | d- 0.6817 | gap 0.1149 | d' 0.569 | 21.0s
Epoch 07/50 | train loss 0.36268 | val loss 0.36039 | val AUC 0.66169 | d+ 0.55

In [25]:
# ============================================================
# CELL 23 — BEST VALIDATION SUMMARY
# ============================================================

best_row = (

    euclidean_v3_history[
        euclidean_v3_history[
            "epoch"
        ]
        ==
        best_epoch
    ]

    .iloc[0]
)


euclidean_v3_summary = pd.DataFrame(
    [
        {
            "loss":
                "Euclidean Contrastive",

            "architecture":
                "V3 bias-free",

            "margin":
                EUCLIDEAN_MARGIN,

            "best_epoch":
                best_epoch,

            "val_auc":
                best_val_auc,

            "val_loss":
                best_row[
                    "val_loss"
                ],

            "d_pos":
                best_row[
                    "d_pos"
                ],

            "d_neg":
                best_row[
                    "d_neg"
                ],

            "gap":
                best_row[
                    "gap"
                ],

            "d_prime":
                best_row[
                    "d_prime"
                ]
        }
    ]
)


display(
    euclidean_v3_summary
)

,loss,architecture,margin,best_epoch,val_auc,val_loss,d_pos,d_neg,gap,d_prime
0,Euclidean Contrastive,V3 bias-free,1.25,38,0.677272,0.353444,0.5535,0.683367,0.129867,0.652263
